# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabeeljohn2/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [33]:
import os, subprocess
import pandas as pd, numpy as np
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("/content/flyrank-ml-internship"):
        subprocess.run(["git","clone","https://github.com/tabeeljohn2/flyrank-ml-internship.git","/content/flyrank-ml-internship"])
    os.chdir("/content/flyrank-ml-internship")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(os.getcwd(), df.shape)

/content/flyrank-ml-internship/flyrank-ml-internship (30000, 44)


In [34]:
for i, c in enumerate(df.columns):
    print(i, c, "|", df[c].dtype, "| missing:", df[c].isna().sum())

0 content_id | object | missing: 0
1 client_id | object | missing: 0
2 search_volume | float64 | missing: 2468
3 competition | float64 | missing: 2468
4 competition_level | object | missing: 2610
5 cpc | float64 | missing: 2468
6 content_type | object | missing: 0
7 main_intent | object | missing: 2374
8 word_count | float64 | missing: 7699
9 char_count | float64 | missing: 7699
10 provider_used | object | missing: 21438
11 model_used | object | missing: 5733
12 impressions_90d | int64 | missing: 0
13 clicks_90d | int64 | missing: 0
14 pageviews_90d | int64 | missing: 0
15 sessions_90d | int64 | missing: 0
16 users_90d | int64 | missing: 0
17 engaged_sessions_90d | int64 | missing: 0
18 ai_sessions_90d | int64 | missing: 0
19 scroll_events_90d | int64 | missing: 0
20 days_with_impressions | int64 | missing: 0
21 days_with_sessions | int64 | missing: 0
22 impressions_last_30d | int64 | missing: 0
23 clicks_last_30d | int64 | missing: 0
24 sessions_last_30d | int64 | missing: 0
25 impres

Column sorting.
IDs: content_id, client_id.
Used in the score: days_since_last_update, impressions_90d.
Not used in the score: trend_direction, trend_pct, impressions_last_30d, impressions_prev_30d.
No column in this file is a label or a product flag.
Lane: Refresh / Content Opportunity Scoring (confirmed, unchanged from ML-03).

In [35]:
def bucket_table(d, x, y, agg="median", q=5):
    d = d.dropna(subset=[x, y]).copy()
    d["bucket"] = pd.qcut(d[x], q, duplicates="drop")
    t = d.groupby("bucket", observed=True).agg(n=(y,"size"), value=(y,agg))
    print(f"{x} vs {y} | total n = {len(d)}")
    return t

In [36]:
d1 = df[df["impressions_prev_30d"] >= 50].copy()
d1["traffic_change"] = d1["impressions_last_30d"] / d1["impressions_prev_30d"]
bucket_table(d1, "days_since_last_update", "traffic_change")

days_since_last_update vs traffic_change | total n = 20249


,n,value
bucket,,
"(4.999, 20.0]",8909,0.594008
"(20.0, 26.0]",3307,0.880952
"(26.0, 104.0]",7918,0.669278
"(104.0, 313.0]",115,0.376552


Signal 1: staleness vs traffic change (flag-linked: refresh flags).
Verdict: MIXED. Median traffic change (last 30d / prev 30d) by staleness bucket was 0.59, 0.88, 0.67, then 0.38 (total n = 20,249). It does not fall steadily. The stalest bucket looks worst but has only n = 115, and the freshest bucket (0.59) is lower than the two middle ones. Staleness is therefore a directional prior in my rule, not a proven driver.

In [37]:
bucket_table(df[df["impressions_90d"] >= 100], "avg_position", "ctr", agg="mean")

avg_position vs ctr | total n = 22006


,n,value
bucket,,
"(0.099, 6.3]",4453,0.404159
"(6.3, 9.6]",4374,0.307387
"(9.6, 15.9]",4400,0.266325
"(15.9, 27.1]",4409,0.207154
"(27.1, 88.9]",4370,0.098931


Signal 2: position vs CTR (the CTR-vs-position logic).
Verdict: CONFIRMED. Mean CTR falls steadily from 0.404 (best positions) to 0.099 (worst positions), with about 4,400 rows per bucket (total n = 22,006). I use this in the top-20 review: a page with a very low CTR for a good position is a CTR problem, not a freshness problem.

## 1. My rule and its reason codes

Rule: pages that have gone a long time without an update and still get many impressions are refreshed first. Score = 0.6 × staleness percentile + 0.4 × impressions percentile (both ranked 0 to 1 across all pages).
Why these weights: Signal 1 was MIXED, so staleness gets only a modest edge over visibility. Visibility is there so we don't send effort to pages nobody sees.

Reason code (one): STALE_HIGH_VIS fires when staleness is in the top 20% and impressions are above the median.
Action: REFRESH if score > 0.75, otherwise MONITOR.

In [38]:
W_STALE, W_VIS = 0.6, 0.4
STALE_CUT = 0.8      # top 20% stalest
ACTION_CUT = 0.75    # score above this = REFRESH


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [39]:
d = df.copy()
d["stale_pct"] = d["days_since_last_update"].rank(pct=True)
d["vis_pct"]   = d["impressions_90d"].rank(pct=True)
d["score"] = W_STALE*d["stale_pct"] + W_VIS*d["vis_pct"]
d["reason_code"] = np.where((d["stale_pct"] > STALE_CUT) & (d["vis_pct"] > 0.5), "STALE_HIGH_VIS", "OTHER")
d["action"] = np.where(d["score"] > ACTION_CUT, "REFRESH", "MONITOR")

queue = d.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

os.makedirs("work/outputs", exist_ok=True)
queue[["rank","client_id","content_id","score","reason_code","action"]].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(queue["action"].value_counts())

action
MONITOR    24790
REFRESH     5210
Name: count, dtype: int64


The score ranks all 30,000 pages. The result was 5,210 REFRESH and 24,790 MONITOR. The ranked queue is written to work/outputs/baseline_action_score.csv by the cell above.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [40]:
cols = ["rank","content_id","score","reason_code","action","days_since_last_update","impressions_90d","avg_position","ctr"]
queue.head(20)[cols]

,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr
0,1,content_cf56e2e2e282,0.992260,STALE_HIGH_VIS,REFRESH,194,61678,19.7,0.15
1,2,content_7368877ea310,0.992100,STALE_HIGH_VIS,REFRESH,194,59472,24.8,0.13
2,3,content_a5dbb404bdc2,0.991180,STALE_HIGH_VIS,REFRESH,106,79035,8.7,0.07
3,4,content_47b8b12d581e,0.985367,STALE_HIGH_VIS,REFRESH,106,40305,28.4,0.96
4,5,content_1bfaa38ff26c,0.979860,STALE_HIGH_VIS,REFRESH,194,25715,22.2,0.23
5,6,content_69fad7e6c50c,0.978740,STALE_HIGH_VIS,REFRESH,106,28000,4.7,1.32
6,7,content_482aff19e9cc,0.977433,STALE_HIGH_VIS,REFRESH,106,26287,13.2,2.43
7,8,content_6ac3ab740bbf,0.974107,STALE_HIGH_VIS,REFRESH,106,22462,4.6,0.14
8,9,content_cb7e312f5d32,0.973940,STALE_HIGH_VIS,REFRESH,151,21272,12.6,2.45
9,10,content_ac1d924c6a70,0.973167,STALE_HIGH_VIS,REFRESH,106,21853,31.7,0.13


**Top-20 review** (observed numbers from the table above; CTR as stored in the data)

**Rank 1, REFRESH:** 194 days since update, 61,678 impressions, position 19.7. Confidence: medium. Wrong if the page is evergreen, or if position ~20 means a refresh won't reach page 1.

**Rank 2, REFRESH:** 194 days, 59,472 impressions, position 24.8. Confidence: medium. Wrong if the impressions come from a broad query the page was never meant to rank for.

**Rank 3, REFRESH:** 106 days, 79,035 impressions, position 8.7 but CTR only 0.07. Confidence: low. Wrong if the problem is the title or snippet, not stale content.

**Rank 4, REFRESH:** 106 days, 40,305 impressions, position 28.4, CTR 0.96. Confidence: low. Wrong if that CTR comes from a few branded queries and the page is actually fine.

**Rank 5, REFRESH:** 194 days, 25,715 impressions, position 22.2. Confidence: medium. Wrong if the update date is a tracking gap and the page was edited recently.

**Rank 6, REFRESH:** 106 days, 28,000 impressions, position 4.7, CTR 1.32. Confidence: low. Wrong if it already performs well and a rewrite risks hurting it.

**Rank 7, REFRESH:** 106 days, 26,287 impressions, position 13.2, CTR 2.43. Confidence: low. Wrong if a CTR this high means the page is healthy and age doesn't matter.

**Rank 8, REFRESH:** 106 days, 22,462 impressions, position 4.6 but CTR 0.14. Confidence: low. Wrong if SERP features or the title are causing the low CTR, which a refresh would not fix.

**Rank 9, REFRESH:** 151 days, 21,272 impressions, position 12.6, CTR 2.45. Confidence: low. Wrong if the page is already healthy, since a CTR this high suggests it is meeting searcher needs.

**Rank 10, REFRESH:** 106 days, 21,853 impressions, position 31.7. Confidence: medium. Wrong if page 3-4 is too deep for a refresh to help and the page needs a full rewrite.

**Rank 11, REFRESH:** 106 days, 16,190 impressions, position 23.9, CTR 0.29. Confidence: medium. Wrong if the impressions come from broad queries and the page is not meant to rank on page 1.

**Rank 12, REFRESH:** 193 days, 13,299 impressions, position 10.5, CTR 0.49. Confidence: medium. Wrong if the page is evergreen and the old update date does not matter.

**Rank 13, REFRESH:** 106 days, 14,099 impressions, position 26.0, CTR 0.45. Confidence: medium. Wrong if the page sits too far down (page 3) for a refresh alone to help.

**Rank 14, REFRESH:** 106 days, 12,029 impressions, position 5.8, CTR 1.01. Confidence: low. Wrong if the page already ranks and performs well, so a rewrite risks hurting it.

**Rank 15, REFRESH:** 106 days, 10,910 impressions, position 37.7, CTR 0.14. Confidence: low. Wrong if position ~38 means the page needs a full rewrite or a new target keyword, not a refresh.

**Rank 16, REFRESH:** 106 days, 10,744 impressions, position 24.4, CTR 0.31. Confidence: medium. Wrong if the update date is a tracking gap and the page was edited recently.

**Rank 17, REFRESH:** 105 days, 10,068 impressions, position 3.6, CTR 0.70. Confidence: low. Wrong if a top-4 position means the page is healthy and staleness is not hurting it.

**Rank 18, REFRESH:** 106 days, 9,594 impressions, position 5.1, CTR 0.75. Confidence: low. Wrong if the page is stable and the traffic is seasonal, not declining.

**Rank 19, REFRESH:** 106 days, 9,582 impressions, position 2.9, CTR 0.49. Confidence: low. Wrong if a position near the top means the page is already winning and needs no refresh.

**Rank 20, REFRESH:** 106 days, 9,188 impressions, position 4.9 but CTR only 0.09. Confidence: low. Wrong if the problem is the title or snippet, which a content refresh would not fix.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [41]:
print("Columns used in score:", ["days_since_last_update","impressions_90d"])
queue.head(50)[cols].describe()
queue.head(50).sort_values("impressions_90d").head(5)[cols]   # smallest-visibility picks


Columns used in score: ['days_since_last_update', 'impressions_90d']


,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr
37,38,content_ecb6215e79fd,0.909000,STALE_HIGH_VIS,REFRESH,194,4429,25.3,0.38
35,36,content_fe16a55cd13d,0.910760,STALE_HIGH_VIS,REFRESH,194,4556,16.4,0.33
34,35,content_b16bd7307b39,0.911200,STALE_HIGH_VIS,REFRESH,194,4590,31.0,0.00
36,37,content_a9e82d7704af,0.909007,STALE_HIGH_VIS,REFRESH,106,4664,27.1,0.32
33,34,content_b9383c4e20d3,0.915907,STALE_HIGH_VIS,REFRESH,106,5286,15.3,0.23


Weak picks: (a) Rank 3: very low CTR (0.07) at position 8.7 suggests a title or snippet problem, not staleness. (b) Ranks 8 and 20: low CTR (0.14 and 0.09) at positions 4.6 and 4.9, same issue. (c) Ranks 34-38: only 4,400-5,300 impressions at positions 15-31, so the upside is small compared with the top ranks.

Leakage check: the score used only days_since_last_update and impressions_90d. No label, product flag, or future-window column went into the score. trend_direction, trend_pct and the 30-day windows were not used in the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅ ] No client names, URLs, or private queries anywhere
- [ ✅ ] My claims use careful words: observed, measured, directional, decision-support
- [ ✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.